In [2]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import time
import json
from urllib.parse import urljoin, urlparse
from collections import deque
from datetime import date
import unicodedata
import re

CUTOFF_DATE = date(2025, 1, 20)

# -----------------------------
# Configuration
# -----------------------------

OUTLET_NAME = "The American Conservative"

# Define MULTIPLE starting URLs
START_URLS = [
    "https://www.theamericanconservative.com/all-articles/",
    "https://www.theamericanconservative.com/feed/",
    "https://www.theamericanconservative.com/",
    "https://www.theamericanconservative.com/category/politics/",
    "https://www.theamericanconservative.com/category/foreign-policy/",
    "https://www.theamericanconservative.com/tag/iran/",
    "https://www.theamericanconservative.com/category/culture/",
    "https://www.theamericanconservative.com/category/uk-special-coverage/",
    "https://www.theamericanconservative.com/issue/march-april-2026/",
    "https://www.theamericanconservative.com/issue/january-february-2026/",
    "https://www.theamericanconservative.com/issue/november-december-2025/",
    "https://www.theamericanconservative.com/issue/september-october-2025/",
    "https://www.theamericanconservative.com/issue/july-august-2025/",
    "https://www.theamericanconservative.com/issue/may-june-2025/",
    "https://www.theamericanconservative.com/issue/march-april-2025/",
    "https://www.theamericanconservative.com/blogs/state_of_the_union/",
]

TARGET_ARTICLE_COUNT = 200  # Stop when we collect this many unique articles
CRAWL_DELAY = 10            # Delay between requests (as per robots.txt)
MIN_WORD_COUNT = 150   # Change to whatever threshold you want

HEADERS = {
    "User-Agent": "AcademicResearchBot/1.0 (LZSCC.413 coursework; contact: e.kolian@lancaster.ac.uk)"
}

# -----------------------------
# Utility helpers
# -----------------------------
def remove_editorial_markup(text: str) -> str:
    """
    Removes tags like [BLOCK]...[/BLOCK], [PULLQUOTE], [FOO], etc.
    Keeps the inner text.
    Works even if tags appear inline or across lines.
    """
    # Remove paired tags: [BLOCK] ... [/BLOCK]
    text = re.sub(r"\[(?:BLOCK|PULLQUOTE|QUOTE|NOTE)\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[/\s*(?:BLOCK|PULLQUOTE|QUOTE|NOTE)\s*\]", "", text, flags=re.IGNORECASE)

    # Remove any other unknown bracketed editorial markers: [SOMETHING]
    # BUT only if they are ALL CAPS (to avoid removing legitimate bracket content)
    text = re.sub(r"\[[A-Z0-9_\-]+\]", "", text)
    
    return text
    
def normalize_text_for_nlp(text):

    if not text:
        return ""

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Remove problematic invisible characters
    text = text.replace("\u00A0", " ")        # NBSP
    text = text.replace("\u2029", "\n\n")     # Paragraph separator (MUST replace)
    text = text.replace("\u2028", "\n")       # Line separator (also MUST replace)
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)  # Zero-width chars

    # Normalize punctuation AFTER unsafe chars removed
    punctuation_map = {
        "\u2018": "'",
        "\u2019": "'",
        "\u02BC": "'",
        "\u2032": "'",
        "\u201C": '"',
        "\u201D": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2026": "...",
    }
    for bad, good in punctuation_map.items():
        text = text.replace(bad, good)

    # Collapse weird spacing
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s+", "\n", text)

    return text.strip()

def is_recent_enough(publication_date_str):
    """
    Return True if publication_date is >= CUTOFF_DATE.
    Expects ISO format (YYYY-MM-DD).
    """
    if not publication_date_str:
        return False  # Skip articles with unknown date

    try:
        pub_date = datetime.strptime(publication_date_str[:10], "%Y-%m-%d").date()
        return pub_date >= CUTOFF_DATE
    except Exception:
        return False
        
def is_article_url(url):
    """Ensure we only crawl actual article URLs."""
    parsed = urlparse(url)
    
    # Must be from the correct domain
    if not parsed.netloc.endswith("theamericanconservative.com"):
        return False
    
    # Should have a path that looks like an article
    if not parsed.path or parsed.path == '/':
        return False
    
    # Exclude specific non-article paths
    excluded_paths = [
        "/wp-", "/feed/", "/about/", "/contact/", "/donate", 
        "/subscribe/", "/advertise/", "/privacy/", "/terms/", 
        "/sitemap/", "/search/", "/login/", "/who-we-are/",
        "/advertise/", "/tac/", "/author/", "/customer-service/",
        "/privacy-policy/",
    ]
    
    for excluded in excluded_paths:
        if excluded in url:
            return False
    
    # Include listing pages (categories, tags, authors, all-articles)
    # These will help us discover more articles
    if any(x in url for x in ["/category/", "/tag/", "/author/", "/all-articles/"]):
        return True
    
    # Should not be a file extension (like .jpg, .pdf, etc.)
    path_lower = parsed.path.lower()
    if any(path_lower.endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.gif', '.pdf', '.zip', '.mp4', '.mp3']):
        return False
    
    # Must be a proper URL with scheme
    if not url.startswith('http'):
        return False
    
    return True

def normalize_url(url):
    """Normalize URL to avoid duplicates."""
    try:
        parsed = urlparse(url)
        # Remove fragments, query parameters, and trailing slashes
        normalized = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
        normalized = normalized.rstrip('/')
        
        # Also normalize www and non-www versions
        if normalized.startswith("https://theamericanconservative.com"):
            normalized = normalized.replace("https://theamericanconservative.com", "https://www.theamericanconservative.com")
        
        return normalized
    except:
        return url

def should_crawl_url(url):
    """Determine if we should crawl this URL."""
    normalized = normalize_url(url)
    
    # Must be from the correct domain
    if not normalized.startswith("https://www.theamericanconservative.com"):
        return False
    
    # Must be an article URL (including listing pages)
    if not is_article_url(normalized):
        return False
    
    # Must not be an obvious non-HTML resource
    if any(normalized.lower().endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.gif', '.pdf', '.zip', '.mp4', '.mp3']):
        return False
    
    return True

# -----------------------------
# Starting URL handler
# -----------------------------

def process_starting_urls(start_urls):
    """Process multiple starting URLs and return initial links."""
    all_links = set()
    
    for start_url in start_urls:
        print(f"Processing starting URL: {start_url}")
        
        try:
            time.sleep(CRAWL_DELAY)
            response = requests.get(start_url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            
            # Check if it's an RSS feed
            if 'xml' in response.headers.get('Content-Type', '').lower() or \
               'rss' in response.headers.get('Content-Type', '').lower() or \
               '<?xml' in response.text[:100] or \
               '<rss' in response.text[:100]:
                
                print("  Detected RSS feed, extracting article links...")
                soup = BeautifulSoup(response.text, "xml")
                for item in soup.find_all("item"):
                    link = item.link.text.strip() if item.link else None
                    if link and should_crawl_url(link):
                        normalized = normalize_url(link)
                        all_links.add(normalized)
                
            else:
                # It's a regular HTML page
                soup = BeautifulSoup(response.text, "html.parser")
                links = extract_all_links(soup, start_url)
                all_links.update(links)
            
            print(f"  Extracted {len(links) if 'links' in locals() else 0} links from this URL")
            
        except Exception as e:
            print(f"  ❌ Failed to process {start_url}: {e}")
    
    return list(all_links)

def extract_all_links(soup, base_url):
    """
    Extract ALL crawlable links from a page.
    This includes article links AND listing page links.
    """
    links = set()
    
    # Extract from ALL anchor tags on the page
    for a in soup.find_all("a", href=True):
        href = a["href"]
        
        # Skip obviously non-crawlable links
        if not href or href.startswith(('#', 'javascript:', 'mailto:', 'tel:')):
            continue
        
        # Convert to absolute URL
        full_url = urljoin(base_url, href)
        normalized = normalize_url(full_url)
        
        # Only add if it's a crawlable URL
        if should_crawl_url(normalized):
            links.add(normalized)
    
    return links

# -----------------------------
# Page fetching and parsing
# -----------------------------

def fetch_page(url):
    """Fetch a page with delay."""
    try:
        time.sleep(CRAWL_DELAY)
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
        return response.text, response.url, True
    except Exception as e:
        print(f"  ❌ Failed to fetch {url}: {e}")
        return None, None, False

def extract_headline(soup):
    """Extract article headline."""
    # Try multiple possible headline locations
    headline_selectors = [
        ("h2", "c-hero-article__title"),
        ("h1", "c-blog-post__title"),
        ("h1", "entry-title"),
        ("h1", None)
    ]
    
    for tag, class_name in headline_selectors:
        if class_name:
            element = soup.find(tag, class_=class_name)
        else:
            element = soup.find(tag)
        if element:
            text = element.get_text(strip=True)
            if text and len(text) > 10:  # Ensure it's a real headline
                return text
    
    return None

def extract_publication_date(soup):
    """Extract publication date."""
    # Try date in byline
    date_span = soup.find("span", class_="o-byline__date-item")
    if date_span:
        date_text = date_span.get_text(strip=True)
        # Try various date formats
        date_formats = [
            "%B %d, %Y",    # "February 3, 2026"
            "%b %d, %Y",    # "Feb 3, 2026"
            "%Y-%m-%d",     # "2026-02-03"
            "%m/%d/%Y",     # "02/03/2026"
        ]
        for date_format in date_formats:
            try:
                return datetime.strptime(date_text, date_format).date().isoformat()
            except ValueError:
                continue
    
    # Try meta tags
    meta_selectors = [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "pubdate"}),
        ("meta", {"name": "publish_date"}),
        ("meta", {"name": "date"}),
    ]
    
    for tag_name, attrs in meta_selectors:
        meta = soup.find(tag_name, attrs)
        if meta and meta.get("content"):
            content = meta["content"]
            # Extract just the date part if it's a full datetime
            if "T" in content:
                return content.split("T")[0]
            return content[:10]
    
    return None

def clean_content(container):
    """Clean content by removing unwanted elements."""
    # Remove script, style, and other non-content elements
    for tag in container.find_all(["script", "iframe", "style", "nav", "aside", "header", "footer", "form"]):
        tag.decompose()
    
    # Remove ads and promotional content
    for tag in container.find_all(class_=lambda x: x and any(
        keyword in str(x).lower() 
        for keyword in ["ad", "promo", "sponsor", "newsletter", "subscription", "popular", "trending"]
    )):
        tag.decompose()

def extract_body_text(soup):
    """Extract main article text."""
    # Try various content containers
    content_selectors = [
        ("div", "c-blog-post__content"),
        ("div", "o-content"),
        ("div", "entry-content"),
        ("article", None),
        ("main", None),
    ]
    
    for tag, class_name in content_selectors:
        if class_name:
            content = soup.find(tag, class_=class_name)
        else:
            content = soup.find(tag)
        
        if content:
            clean_content(content)
            
            # Extract text from paragraphs
            paragraphs = []
            for p in content.find_all("p"):
                text = p.get_text(" ", strip=True)
                if len(text) > 50:  # Only include substantial paragraphs
                    paragraphs.append(text)
            
            if paragraphs:
                return "\n\n".join(paragraphs)
    
    return ""

def is_actual_article_page(soup):
    """Check if the page is an actual article (not a listing)."""
    # Look for article-specific elements
    article_indicators = [
        soup.find("div", class_="c-blog-post__content"),
        soup.find("div", class_="o-content"),
        soup.find("div", class_="entry-content"),
        soup.find("article"),
        soup.find("span", class_="o-byline__date-item"),
        soup.find("h1", class_="c-blog-post__title"),
        soup.find("h2", class_="c-hero-article__title"),
    ]
    
    # If any article indicator is found, it's an article page
    return any(indicator for indicator in article_indicators if indicator)

def parse_page(html, url):
    """
    Parse a page and extract article data and links.
    Returns: (article_data, links)
    """
    soup = BeautifulSoup(html, "html.parser")

    # Determine if this is an article page
    is_article_page = is_actual_article_page(soup)
    article_data = None

    if is_article_page:

        # Extract components
        raw_headline = extract_headline(soup)
        headline = normalize_text_for_nlp(raw_headline)
    
        # Extract raw body text
        raw_body = extract_body_text(soup)
    
        # First remove editorial markup, THEN normalize
        cleaned_body = remove_editorial_markup(raw_body)
        body_text = normalize_text_for_nlp(cleaned_body)
    
        publication_date = extract_publication_date(soup)
    
        word_count = len(body_text.split()) if body_text else 0
    
        # Apply filters
        if (
            headline
            and publication_date
            and is_recent_enough(publication_date)
            and word_count >= MIN_WORD_COUNT
        ):
            article_data = {
                "article_id": hash(url),
                "url": url,
                "headline": headline,
                "body_text": body_text,
                "publication_date": publication_date,
                "outlet_name": OUTLET_NAME,
                "word_count": word_count,
                "label": "right",
                "crawl_timestamp": datetime.now().isoformat(),
                "page_type": "article"
            }
        else:
            article_data = {
                "url": url,
                "page_type": "article",
                "error": "filtered_out",
                "headline": headline or "Unknown",
                "publication_date": publication_date,
                "word_count": word_count,
                "crawl_timestamp": datetime.now().isoformat()
            }

    else:
        # Listing or non-article page
        article_data = {
            "url": url,
            "page_type": "listing",
            "crawl_timestamp": datetime.now().isoformat()
        }

    # Extract all links from page
    links = extract_all_links(soup, url)

    return article_data, list(links)
    
# -----------------------------
# Main crawler
# -----------------------------

def crawl_website():
    """
    Crawl ONLY the URLs extracted from START_URLS.
    No BFS.
    No adding additional URLs from parsed pages.
    Stops when TARGET_ARTICLE_COUNT is reached.
    """
    visited = set()
    articles = []
    stats = {
        "total_pages_visited": 0,
        "article_pages_visited": 0,
        "listing_pages_visited": 0,
        "failed_pages": 0,
        "unique_articles_found": 0,
        "start_time": time.time()
    }

    print("=" * 70)
    print(f"🚀 Starting automated crawl of {OUTLET_NAME}")
    print(f"🎯 Target: {TARGET_ARTICLE_COUNT} unique articles")
    print(f"🔗 Starting URLs: {len(START_URLS)}")
    print("=" * 70)

    # Step 1: Process starting URLs
    print("\n📂 Processing starting URLs...")
    starting_links = process_starting_urls(START_URLS)

    print(f"\n✅ Extracted {len(starting_links)} unique URLs from starting pages")

    # Step 2: Visit each extracted link ONCE (no BFS)
    for url in starting_links:

        if len(articles) >= TARGET_ARTICLE_COUNT:
            print(f"\n🎯 TARGET REACHED: Collected {len(articles)} articles!")
            break

        if url in visited:
            continue

        visited.add(url)
        stats["total_pages_visited"] += 1

        progress = f"[Page {stats['total_pages_visited']}] Articles: {len(articles)}/{TARGET_ARTICLE_COUNT}"
        print(f"\n{progress} Processing: {url[:80]}...")

        html, final_url, success = fetch_page(url)
        if not success:
            stats["failed_pages"] += 1
            continue

        try:
            page_data, _ = parse_page(html, final_url)

            if page_data.get("page_type") == "article":
                stats["article_pages_visited"] += 1

                if "body_text" in page_data and len(page_data["body_text"]) > 300:
                    existing_urls = {article['url'] for article in articles}
                    if page_data['url'] not in existing_urls:
                        articles.append(page_data)
                        stats["unique_articles_found"] += 1
                        print(f"  ✅ Article added: {page_data['headline'][:60]}...")
                        print(f"     Words: {page_data['word_count']}, Date: {page_data['publication_date'] or 'Unknown'}")
                    else:
                        print("  ⚠️ Article already collected")
                else:
                    print("  📄 Article page (insufficient content)")
            else:
                stats["listing_pages_visited"] += 1
                print("  📚 Listing page")

        except Exception as e:
            print(f"  ❌ Error parsing page: {e}")
            stats["failed_pages"] += 1

    # Final statistics
    elapsed_time = time.time() - stats["start_time"]
    stats["elapsed_time_seconds"] = elapsed_time
    stats["articles_collected"] = len(articles)

    if articles:
        stats["total_words"] = sum(article.get('word_count', 0) for article in articles)
        stats["average_words_per_article"] = stats["total_words"] / len(articles)
    else:
        stats["total_words"] = 0
        stats["average_words_per_article"] = 0

    print("\n" + "=" * 70)
    print("CRAWL COMPLETED - FINAL STATISTICS")
    print("=" * 70)
    print(f"Total pages visited: {stats['total_pages_visited']}")
    print(f"Article pages visited: {stats['article_pages_visited']}")
    print(f"Listing pages visited: {stats['listing_pages_visited']}")
    print(f"Failed pages: {stats['failed_pages']}")
    print(f"Articles collected: {stats['articles_collected']}")
    print(f"Total words collected: {stats['total_words']}")
    print(f"Average words per article: {stats['average_words_per_article']:.0f}")
    print(f"Elapsed time: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
    print(f"Average time per page: {elapsed_time/stats['total_pages_visited']:.1f}s" if stats['total_pages_visited'] > 0 else "No pages visited")

    return articles, stats
# -----------------------------
# Save functions
# -----------------------------

def save_articles(articles, stats, filename=None):
    """Save ONLY clean article blocks (no metadata wrapper)."""
    
    if not filename:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{OUTLET_NAME.replace(' ', '_').lower()}_{len(articles)}_articles_{timestamp}.json"
    
    # Keep only valid article entries
    cleaned_articles = []

    for article in articles:
        if "body_text" not in article or article.get("word_count", 0) == 0:
            continue

        cleaned_articles.append({
            "article_id": article.get("article_id") or hash(article["url"]),
            "url": article["url"],
            "headline": article.get("headline"),
            "body_text": article.get("body_text"),
            "publication_date": article.get("publication_date"),
            "outlet_name": article.get("outlet_name"),
            "word_count": article.get("word_count"),
            "label": article.get("label"),
            "crawl_timestamp": article.get("crawl_timestamp"),
        })

    # ✅ Save as PURE list
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(cleaned_articles, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Saved {len(cleaned_articles)} clean articles to {filename}")
    
# -----------------------------
# Entry point
# -----------------------------

if __name__ == "__main__":
    # Run automatically without user input
    print("🤖 Running in automatic mode...")
    print("📈 Will crawl until target article count is reached.")
    print("⏳ This may take some time depending on the target count and website structure.\n")
    
    try:
        # Start the crawl
        articles, stats = crawl_website()
        
        # Save the results
        if articles:
            save_articles(articles, stats)
        else:
            print("\n❌ No articles were collected!")
            
    except KeyboardInterrupt:
        print("\n\n⚠️ Crawling interrupted by user!")
        
        # Save what we have if interrupted
        if 'articles' in locals() and articles:
            print(f"\n💾 Saving {len(articles)} collected articles before exit...")
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            save_articles(articles, {}, f"interrupted_crawl_{timestamp}.json")
            
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
        
        # Try to save what we have
        if 'articles' in locals() and articles:
            print(f"\n💾 Attempting to save {len(articles)} collected articles...")
            try:
                save_articles(articles, {}, "error_crawl.json")
            except:
                print("Could not save articles")

🤖 Running in automatic mode...
📈 Will crawl until target article count is reached.
⏳ This may take some time depending on the target count and website structure.

🚀 Starting automated crawl of The American Conservative
🎯 Target: 200 unique articles
🔗 Starting URLs: 16

📂 Processing starting URLs...
Processing starting URL: https://www.theamericanconservative.com/all-articles/
  Extracted 50 links from this URL
Processing starting URL: https://www.theamericanconservative.com/feed/
  Detected RSS feed, extracting article links...
  Extracted 50 links from this URL
Processing starting URL: https://www.theamericanconservative.com/
  Extracted 53 links from this URL
Processing starting URL: https://www.theamericanconservative.com/category/politics/
  Extracted 40 links from this URL
Processing starting URL: https://www.theamericanconservative.com/category/foreign-policy/
  Extracted 40 links from this URL
Processing starting URL: https://www.theamericanconservative.com/tag/iran/
  Extracted